# University Regulation RAG Assistant
## Google Colab Generative AI Project

**Selected topic:** University Regulation RAG Assistant

Pipeline: PDF upload → text extraction → chunking → embeddings → FAISS semantic search → similarity threshold → grounded generation → document/page citation.

The system answers only from the uploaded university documents and reports when supporting information is not found.


In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate gradio

In [ ]:
import os, re
import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from google.colab import files


## 1. Upload multiple PDF documents

Upload university regulations, examination rules, attendance policies, student handbooks, curriculum rules, etc.


In [ ]:
uploaded = files.upload()
pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
print("Uploaded PDFs:", pdf_paths)


Saving R22 Regulations for B.Tech.pdf to R22 Regulations for B.Tech (1).pdf
Uploaded PDFs: ['R22 Regulations for B.Tech (1).pdf']


## 2. Extract PDF text with source metadata

Every extracted page keeps its document name and page number so that answers can cite their source.


In [ ]:
def extract_pages(pdf_paths):
    records = []
    for pdf_path in pdf_paths:
        reader = PdfReader(pdf_path)
        for page_number, page in enumerate(reader.pages, start=1):
            text = re.sub(r"\\s+", " ", page.extract_text() or "").strip()
            if text:
                records.append({
                    "text": text,
                    "source": os.path.basename(pdf_path),
                    "page": page_number
                })
    return records

page_records = extract_pages(pdf_paths)
print(f"Extracted {len(page_records)} non-empty pages.")
display(pd.DataFrame(page_records))

Extracted 38 non-empty pages.


,text,source,page
0,Academic\nRegulationsR22\nIn Compliance with N...,R22 Regulations for B.Tech (1).pdf,1
1,PREFACE\n‘You are born to Blossom’ – What an i...,R22 Regulations for B.Tech (1).pdf,3
2,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,4
3,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,5
4,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,7
5,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,8
6,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,9
7,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,10
8,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,11
9,4\nR22 B.Tech.\nYear\nDegree \nProgramme\nVFST...,R22 Regulations for B.Tech (1).pdf,12


## 3. Chunking

We use 500-character chunks with 100-character overlap. Overlap helps preserve information that crosses chunk boundaries.


In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

def chunk_text(text, chunk_size=500, overlap=100):
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        part = text[start:end].strip()
        if part:
            chunks.append(part)
        if end >= len(text):
            break
        start = end - overlap
    return chunks

chunks = []
for rec in page_records:
    for part in chunk_text(rec["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({
            "chunk_id": len(chunks),
            "text": part,
            "source": rec["source"],
            "page": rec["page"]
        })

print("Total chunks:", len(chunks))
display(pd.DataFrame(chunks).head())


Total chunks: 280


,chunk_id,text,source,page
0,0,Academic\nRegulationsR22\nIn Compliance with N...,R22 Regulations for B.Tech (1).pdf,1
1,1,PREFACE\n‘You are born to Blossom’ – What an i...,R22 Regulations for B.Tech (1).pdf,3
2,2,"book cited above, Honourable Kalam, Former Pre...",R22 Regulations for B.Tech (1).pdf,3
3,3,"policies laid down by the University, \nto rea...",R22 Regulations for B.Tech (1).pdf,3
4,4,cepts of the policies brought out in National ...,R22 Regulations for B.Tech (1).pdf,3


## 4. Embeddings + FAISS vector store

`all-MiniLM-L6-v2` converts chunks into vectors. We normalize vectors and use FAISS inner-product search, which corresponds to cosine similarity for normalized vectors.


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

texts = [x["text"] for x in chunks]
embeddings = embedder.encode(
    texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embedding dimension:", embeddings.shape[1])
print("Vectors in FAISS:", index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding dimension: 384
Vectors in FAISS: 280


## 5. Semantic retrieval

`TOP_K` controls retrieval depth. `SIMILARITY_THRESHOLD` prevents weak matches from being sent to the generator.


In [ ]:
TOP_K = 5
SIMILARITY_THRESHOLD = 0.30

def retrieve(query, top_k=TOP_K, threshold=SIMILARITY_THRESHOLD):
    q = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q, top_k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0 and float(score) >= threshold:
            item = chunks[int(idx)].copy()
            item["score"] = float(score)
            results.append(item)
    return results

results = retrieve("What is the minimum attendance required for examinations?")
for r in results:
    print(f"[{r['score']:.3f}] {r['source']} — page {r['page']}")
    print(r["text"])
    print("-"*70)


[0.645] R22 Regulations for B.Tech (1).pdf — page 17
n the ratio of 60:40, respectively.
5.2 Qualifying criteria
 To be declared successful in a course, a student must secure at least a grade 4.0 in a scale 
of 10 based on the total maximum marks which is inclusive of formative and summative 
assessment. The students should also get 35% from the maximum marks allotted for formative 
and summative assessments individually.
 The hierarchy of qualifying criteria is as follows:
i. attendance compliance should be 75% or within condonable range; else th
----------------------------------------------------------------------
[0.575] R22 Regulations for B.Tech (1).pdf — page 27
.00 to 6.99 good b
≥ 5.00 to 5.99 Fair C
≥ 4.00 to 4.99 Marginal M
Transitional grade Repeat R
Transitional grade Incomplete I
8. suPPleMentaRY eXaMinations
8.1 The supplementary examinations shall be conducted once in summer semester. Notifications 
will be released by the examination section informing the students abou

## 6. Local grounded generation model

FLAN-T5 runs in Colab without requiring a paid API. The prompt instructs it to use only retrieved context.


In [ ]:
GENERATION_MODEL = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL)

def custom_generator(prompt_text, max_new_tokens=220, do_sample=False):
    input_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    outputs = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=do_sample)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": generated_text}]

generator = custom_generator
print("Generator ready.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generator ready.


## 7. Complete RAG pipeline

If no retrieved chunk crosses the threshold, the system returns an explicit "Information not found" message.


In [ ]:
def build_context(results):
    return "\n\n".join(
        f"[SOURCE {i}] Document: {r['source']} | Page: {r['page']}\n{r['text']}"
        for i, r in enumerate(results, 1)
    )

def answer_question(question):
    question = question.strip()
    if not question:
        return "Please enter a question.", []

    results = retrieve(question)
    if not results:
        return (
            "Information not found in the uploaded university documents.",
            []
        )

    context = build_context(results)
    prompt = """You are a university regulation question-answering assistant.

Answer the QUESTION using ONLY the CONTEXT.
Do not use outside knowledge.
Do not invent rules, dates, numbers, requirements, or exceptions.
If the context does not clearly contain the answer, say:
Information not found in the uploaded university documents.

QUESTION:
%s

CONTEXT:
%s

ANSWER:""" % (question, context)

    answer = generator(prompt)[0]["generated_text"].strip()

    citations = []
    seen = set()
    for r in results:
        key = (r["source"], r["page"])
        if key not in seen:
            citations.append({
                "document": r["source"],
                "page": r["page"],
                "similarity": round(r["score"], 3)
            })
            seen.add(key)
    return answer, citations


In [ ]:
question = "What is the minimum attendance required for the end-semester examination?"
answer, citations = answer_question(question)

print("QUESTION:", question)
print("\nANSWER:", answer)
print("\nSOURCES:")
for c in citations:
    print(f"- {c['document']} — Page {c['page']} — similarity {c['similarity']}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (808 > 512). Running this sequence through the model will result in indexing errors


QUESTION: What is the minimum attendance required for the end-semester examination?

ANSWER: 75%

SOURCES:
- R22 Regulations for B.Tech (1).pdf — Page 17 — similarity 0.614
- R22 Regulations for B.Tech (1).pdf — Page 8 — similarity 0.571
- R22 Regulations for B.Tech (1).pdf — Page 16 — similarity 0.539
- R22 Regulations for B.Tech (1).pdf — Page 27 — similarity 0.539


## 8. Gradio chatbot interface

Run this cell to demonstrate the finished project.


In [ ]:
import gradio as gr

def chat_fn(question):
    answer, citations = answer_question(question)
    if citations:
        source_text = "\n".join(
            f"• {c['document']} — Page {c['page']} — similarity {c['similarity']}"
            for c in citations
        )
    else:
        source_text = "No supporting source was found."
    return answer, source_text

demo = gr.Interface(
    fn=chat_fn,
    inputs=gr.Textbox(
        label="Ask about university regulations",
        placeholder="Example: What is the minimum attendance required?"
    ),
    outputs=[
        gr.Textbox(label="Grounded Answer"),
        gr.Textbox(label="Source Document / Page")
    ],
    title="University Regulation RAG Assistant",
    description="Answers are generated from the uploaded university PDFs only.",
    examples=[
        ["What is the minimum attendance required for examinations?"],
        ["What documents must students carry to the examination hall?"],
        ["What are the credits for a course?"],
        ["What is the attendance requirement?"]
    ]
)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36cfea5a2606376800.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
